In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc matplotlib numpy qiskit-addon-mpf qiskit-addon-utils scipy
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     channel="ibm_quantum_platform",
#     token="<your-api-key>",
#     # instance="<IBM Cloud CRN or instance name>",  # optional
#     set_as_default=True,
#     overwrite=True,
# )

# Mga multi-product formula para mabawasan ang Trotter error

import TutorialFeedback from '@site/src/components/TutorialFeedback';






*Tinatayang paggamit ng QPU: Apat na minuto sa isang Heron r2 processor (TANDAAN: Tantiya lamang ito. Maaaring mag-iba ang iyong runtime.)*

## Pangkalahatang-ideya
Ipinapakita ng tutorial na ito kung paano gamitin ang isang Multi-Product Formula (MPF) upang makamit ang mas mababang Trotter error sa ating observable kumpara sa error na maidudulot ng pinakamalalim na Trotter circuit na ating isasagawa.
Binabawasan ng mga MPF ang Trotter error ng Hamiltonian dynamics sa pamamagitan ng may-timbang na kombinasyon ng ilang circuit execution. Isaalang-alang ang gawain ng paghahanap ng mga expectation value ng observable para sa quantum state
$\rho(t)=e^{-i H t} \rho(0) e^{i H t}$ na may Hamiltonian na $H$. Maaaring gumamit ng mga Product Formula (PF) upang tantiyahin ang time-evolution na $e^{-i H t}$ sa pamamagitan ng sumusunod:

- Isulat ang Hamiltonian na $H$ bilang $H=\sum_{a=1}^d F_a,$ kung saan ang $F_a$ ay mga Hermitian operator na ang bawat katumbas na unitary ay maaaring mahusay na maipatupad sa isang quantum device.
- Tantiyahin ang mga terminong $F_a$ na hindi nag-commute sa isa't isa.

Kaya naman, ang first-order PF (Lie-Trotter formula) ay:

$$S_1(t):=\prod_{a=1}^d e^{-i F_a t},$$

na may quadratic error term na $S_1(t)=e^{-i H t}+\mathcal{O}\left(t^{2}\right)$. Maaari ring gumamit ng higher-order PF (Lie-Trotter-Suzuki formula), na mas mabilis na magkakalapit, at tinukoy nang rekursibo bilang:

$$S_2(t):=\prod_{a=1}^d e^{-i F_a t/2}\prod_{a=1}^d e^{-i F_a t/2}$$

$$S_{2 \chi}(t):= S_{2 \chi -2}(s_{\chi}t)^2 S_{2 \chi -2}((1-4s_{\chi})t)S_{2 \chi -2}(s_{\chi}t)^2,$$

kung saan ang $\chi$ ay ang order ng symmetric PF at $s_p = \left( 4 - 4^{1/(2p-1)} \right)^{-1}$. Para sa matagalang time-evolution, maaaring hatiin ang time interval na $t$ sa $k$ na interval, na tinatawag na mga Trotter step, na may tagal na $t/k$ at tantiyahin ang time-evolution sa bawat interval gamit ang $\chi$ order product formula na $S_{\chi}$. Kaya naman, ang PF ng order na $\chi$ para sa time-evolution operator sa $k$ Trotter step ay:

$$ S_{\chi}^{k}(t) = \left[ S_{\chi} \left( \frac{t}{k} \right)\right]^k = e^{-i H t}+O\left(t \left( \frac{t}{k} \right)^{\chi} \right)$$

kung saan ang error term ay bumababa kasabay ng bilang ng mga Trotter step na $k$ at ng order na $\chi$ ng PF.

Kapag may ibinigay na integer na $k \geq 1$ at isang product formula na $S_{\chi}(t)$, ang tinantyang time-evolved state na $\rho_k(t)$ ay maaaring makuha mula sa $\rho_0$ sa pamamagitan ng pag-apply ng $k$ ulit ng product formula na $S_{\chi}\left(\frac{t}{k}\right)$.

$$
\rho_k(t)=S_{\chi}\left(\frac{t}{k}\right)^k \rho_0 S_{\chi}\left(\frac{t}{k}\right)^{-k}
$$

Ang $\rho_k(t)$ ay isang tantiya para sa $\rho(t)$ na may Trotter approximation error na ||$\rho_k(t)-\rho(t) ||$. Kung isasaalang-alang natin ang isang linear na kombinasyon ng mga Trotter approximation ng $\rho(t)$:

$$
\mu(t) = \sum_{j}^{l} x_j \rho^{k_j}_{j}\left(\frac{t}{k_j}\right) + \text{some remaining Trotter error} \, ,
$$

kung saan ang $x_j$ ay ang ating mga weighting coefficient, ang $\rho^{k_j}_j$ ay ang density matrix na katumbas ng pure state na nakuha sa pamamagitan ng pag-evolve ng initial state gamit ang product formula, $S^{k_j}_{\chi}$, na nagsasangkot ng $k_j$ Trotter step, at ang $j \in {1, ..., l}$ ay nag-i-index ng bilang ng mga PF na bumubuo sa MPF. Lahat ng termino sa $\mu(t)$ ay gumagamit ng parehong product formula na $S_{\chi}(t)$ bilang basehan.
Ang layunin ay mapabuti ang ||$\rho_k(t)-\rho(t) \|$ sa pamamagitan ng paghahanap ng $\mu(t)$ na may mas mababang $\|\mu(t)-\rho(t)\|$.

* Hindi kailangang maging pisikal na estado ang $\mu(t)$ dahil hindi kailangang positibo ang $x_i$. Ang layunin dito ay mabawasan ang error sa expectation value ng mga observable at hindi upang makahanap ng pisikal na kapalit para sa $\rho(t)$.
* Tinutukoy ng $k_j$ ang parehong circuit depth at antas ng Trotter approximation. Ang mas maliit na halaga ng $k_j$ ay nagbubunga ng mas maikling mga circuit, na nagdudulot ng mas kaunting circuit error ngunit magiging mas hindi tumpak na tantiya ng nais na estado.

Ang susi dito ay ang natitirang Trotter error na ibinibigay ng $\mu(t)$ ay mas maliit kaysa sa Trotter error na makukuha kung gagamitin lamang ang pinakamataas na halaga ng $k_j$.

Maaaring tingnan ang kapaki-pakinabang nito mula sa dalawang perspektibo:

1. Para sa isang nakapirming badyet ng Trotter step na maisasagawa, makakakuha ng mga resulta na may mas maliit na Trotter error sa kabuuan.
2. Kung may target na bilang ng Trotter step na masyadong malaki upang maisagawa, maaaring gamitin ang MPF upang makahanap ng isang koleksyon ng mas mababang-lalim na mga circuit na nagbubunga ng katulad na Trotter error.
## Mga Kinakailangan
Bago simulan ang tutorial na ito, tiyakin na naka-install ang mga sumusunod:

* Qiskit SDK v1.0 o mas bago, na may suporta sa [visualization](https://docs.quantum.ibm.com/api/qiskit/visualization)
* Qiskit Runtime v0.22 o mas bago (`pip install qiskit-ibm-runtime`)
* MPF Qiskit addons (`pip install qiskit_addon_mpf`)
* Qiskit addons utils (`pip install qiskit_addon_utils`)
* Quimb library (`pip install quimb`)
* Qiskit Quimb library (`pip install qiskit-quimb`)
* Numpy v0.21 para sa compatibility sa iba't ibang package (`pip install numpy==0.21`)
## Bahagi I. Maliit na halimbawa
### Suriin ang katatagan ng MPF
Walang malinaw na limitasyon sa pagpili ng bilang ng mga Trotter step na $k_j$ na bumubuo sa MPF state na $\mu(t)$. Gayunpaman, dapat itong maingat na piliin upang maiwasan ang kawalang-tatag sa mga expectation value na kinakalkula mula sa $\mu(t)$. Ang isang magandang pangkalahatang tuntunin ay itakda ang pinakamaliit na Trotter step na $k_{\text{min}}$ upang $t/k_{\text{min}} \lt 1$. Kung nais mong matuto pa tungkol dito at kung paano pipiliin ang iyong iba pang $k_j$ na halaga, sumangguni sa gabay na [How to choose the Trotter steps for an MPF](https://qiskit.github.io/qiskit-addon-mpf/how_tos/choose_trotter_steps.html).

Sa halimbawa sa ibaba, sinusuri natin ang katatagan ng MPF solution sa pamamagitan ng pagkalkula ng expectation value ng magnetisasyon para sa isang hanay ng mga oras gamit ang iba't ibang time-evolved state. Partikular, inihahambing natin ang mga expectation value na kinakalkula mula sa bawat isa sa mga tinantyang time-evolution na isinagawa gamit ang katumbas na Trotter step at ang iba't ibang MPF model (static at dynamic na coefficient) kasama ang eksaktong halaga ng time-evolved observable. Una, tukuyin natin ang mga parameter para sa mga Trotter formula at ang mga oras ng evolution.

In [1]:
import warnings

import numpy as np
import matplotlib.pyplot as plt
from functools import partial
from copy import deepcopy

from qiskit import QuantumCircuit
from qiskit.quantum_info import Pauli, SparsePauliOp, Statevector
from qiskit.synthesis import SuzukiTrotter
from qiskit.transpiler import CouplingMap, PassManager
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
from qiskit.circuit.library import XXPlusYYGate
from qiskit.transpiler.passes.optimization.collect_and_collapse import (
    CollectAndCollapse,
    collect_using_filter_function,
    collapse_to_operation,
)

from qiskit_aer import AerSimulator
from qiskit_ibm_runtime import EstimatorV2 as Estimator, QiskitRuntimeService

from qiskit_addon_utils.problem_generators import (
    generate_xyz_hamiltonian,
    generate_time_evolution_circuit,
)
from qiskit_addon_utils.slicing import slice_by_depth
from qiskit_addon_mpf.static import setup_static_lse
from qiskit_addon_mpf.dynamic import setup_dynamic_lse
from qiskit_addon_mpf.costs import (
    setup_exact_problem,
    setup_sum_of_squares_problem,
    setup_frobenius_problem,
)
from qiskit_addon_mpf.backends.tenpy_layers import (
    LayerModel,
    LayerwiseEvolver,
)
from qiskit_addon_mpf.backends.tenpy_tebd import MPOState, MPS_neel_state

from scipy.linalg import expm

# Suppress TeNPy's `unit_cell_width` future-API warning. The default
# (`unit_cell_width=len(sites)`) is correct for Chain lattices, which is what
# `CouplingMap.from_line(...)` produces here, so the warning is informational.
warnings.filterwarnings(
    "ignore",
    message=r".*unit_cell_width.*",
    category=UserWarning,
)


# --- Helper: collect XX + YY rotations into a single gate ---
def filter_function(node):
    return node.op.name in {"rxx", "ryy"}


collect_function = partial(
    collect_using_filter_function,
    filter_function=filter_function,
    split_blocks=True,
    min_block_size=1,
)


def collapse_to_xx_plus_yy(block):
    param = 0.0
    for node in block.data:
        param += node.operation.params[0]
    return XXPlusYYGate(param)


collapse_function = partial(
    collapse_to_operation,
    collapse_function=collapse_to_xx_plus_yy,
)

pm = PassManager()
pm.append(CollectAndCollapse(collect_function, collapse_function))

Para sa halimbawang ito, gagamitin natin ang Neel state bilang initial state na $\vert \text{Neel} \rangle = \vert 0101...01 \rangle$ at ang Heisenberg model sa isang linya ng 10 site para sa Hamiltonian na namamahala sa time-evolution

$$
\hat{\mathcal{H}}_{Heis} = J \sum_{i=1}^{L-1} \left(X_i X_{(i+1)}+Y_i Y_{(i+1)}+ Z_i Z_{(i+1)} \right) \, ,
$$

kung saan ang $J$ ay ang coupling strength para sa mga pinakamalapit na gilid.

In [2]:
L = 10

# Generate coupling map and Hamiltonian
coupling_map = CouplingMap.from_line(L, bidirectional=False)

hamiltonian = generate_xyz_hamiltonian(
    coupling_map,
    coupling_constants=(1.0, 1.0, 1.0),
    ext_magnetic_field=(0.0, 0.0, 0.0),
)
print(hamiltonian)

SparsePauliOp(['IIIIIIIXXI', 'IIIIIIIYYI', 'IIIIIIIZZI', 'IIIIIXXIII', 'IIIIIYYIII', 'IIIIIZZIII', 'IIIXXIIIII', 'IIIYYIIIII', 'IIIZZIIIII', 'IXXIIIIIII', 'IYYIIIIIII', 'IZZIIIIIII', 'IIIIIIIIXX', 'IIIIIIIIYY', 'IIIIIIIIZZ', 'IIIIIIXXII', 'IIIIIIYYII', 'IIIIIIZZII', 'IIIIXXIIII', 'IIIIYYIIII', 'IIIIZZIIII', 'IIXXIIIIII', 'IIYYIIIIII', 'IIZZIIIIII', 'XXIIIIIIII', 'YYIIIIIIII', 'ZZIIIIIIII'],
              coeffs=[1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j,
 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j,
 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j])


In [3]:
# Observable: ZZ on the middle pair of qubits
observable = SparsePauliOp.from_sparse_list(
    [("ZZ", (L // 2 - 1, L // 2), 1.0)], num_qubits=L
)
print(observable)

SparsePauliOp(['IIIIZZIIII'],
              coeffs=[1.+0.j])


In [4]:
# MPF parameters
mpf_trotter_steps = [1, 2, 4]
order = 2
symmetric = False

trotter_times = np.arange(0.5, 1.55, 0.1)
exact_evolution_times = np.arange(trotter_times[0], 1.55, 0.05)

#### Build Trotter circuits

We create the circuits implementing the approximate Trotter time-evolutions for each time point and each Trotter step count. The `CollectAndCollapse` pass defined in the Setup section collects XX and YY rotations into single XX+YY gates, to prepare for more efficient tensor-network simulation later.

In [5]:
# Initial Neel state preparation
initial_state_circ = QuantumCircuit(L)
initial_state_circ.x([i for i in range(L) if i % 2 != 0])


all_circs = []
for total_time in trotter_times:
    mpf_trotter_circs = [
        generate_time_evolution_circuit(
            hamiltonian,
            time=total_time,
            synthesis=SuzukiTrotter(reps=num_steps, order=order),
        )
        for num_steps in mpf_trotter_steps
    ]

    mpf_trotter_circs = pm.run(
        mpf_trotter_circs
    )  # Collect XX and YY into XX + YY

    mpf_circuits = [
        initial_state_circ.compose(circuit) for circuit in mpf_trotter_circs
    ]
    all_circs.append(mpf_circuits)

In [6]:
mpf_circuits[-1].draw("mpl", fold=-1)

<Image src="../docs/images/tutorials/multi-product-formula/extracted-outputs/c7ee61e7-0.avif" alt="Output of the previous code cell" />

Pagkatapos ay lilikha tayo ng mga circuit na nagpapatupad ng mga tinantyang Trotter time-evolution.

In [7]:
aer_sim = AerSimulator()
pm_sim = generate_preset_pass_manager(backend=aer_sim, optimization_level=3)

isa_circs_all_times = [
    pm_sim.run([deepcopy(c) for c in mpf_circuits])
    for mpf_circuits in all_circs
]

### Step 3: Execute using Qiskit primitives

For the small-scale example we run the ISA-lowered Trotter circuits through the `EstimatorV2` primitive backed by Aer. Doing so gives us a *noiseless* reference value for each $(k_j, t)$ pair &mdash; these are the $\langle A \rangle_{k_j}(t)$ values that the MPF will combine in Step 4. We sweep over evolution times so that we can later plot the full time-series curve of each individual product formula and of the MPF.

In [8]:
estimator = Estimator(mode=aer_sim)

mpf_expvals_all_times, mpf_stds_all_times = [], []
for isa_circuits in isa_circs_all_times:
    result = estimator.run(
        [(circuit, observable) for circuit in isa_circuits], precision=0.005
    ).result()
    mpf_expvals_all_times.append([res.data.evs for res in result])
    mpf_stds_all_times.append([res.data.stds for res in result])

![Output of the previous code cell](../docs/images/tutorials/multi-product-formula/extracted-outputs/92dc20a7-0.avif)

Susunod, kinakalkula natin ang mga time-evolved na expectation value mula sa mga Trotter circuit.

In [9]:
exact_expvals = []
for t in exact_evolution_times:
    exp_H = expm(-1j * t * hamiltonian.to_matrix())
    initial_state = Statevector(initial_state_circ).data
    time_evolved_state = exp_H @ initial_state

    exact_obs = (
        time_evolved_state.conj()
        @ observable.to_matrix()
        @ time_evolved_state
    ).real
    exact_expvals.append(exact_obs)

Kinakalkula rin natin ang mga eksaktong expectation value para sa paghahambing.

In [10]:
lse = setup_static_lse(mpf_trotter_steps, order=order, symmetric=symmetric)

#### Mga static na MPF coefficient
Ang mga static MPF ay ang mga kung saan ang mga halaga ng $x_j$ ay hindi nakasalalay sa oras ng evolution na $t$. Isaalang-alang ang order na $\chi = 1$ PF na may $k_j$ Trotter step, maaari itong isulat bilang:

$$ S_1^{k_j}\left( \frac{t}{k_j} \right)=e^{-i H t}+ \sum_{n=1}^{\infty} A_n \frac{t^{n+1}}{k_j^n}  $$

kung saan ang $A_n$ ay mga matrix na nakasalalay sa mga commutator ng mga $F_a$ na termino sa decomposition ng Hamiltonian. Mahalagang tandaan na ang $A_n$ mismo ay independyente sa oras at sa bilang ng mga Trotter step na $k_j$. Kaya naman, posibleng kanselahin ang mga lower-order error term na nag-aambag sa $\mu(t)$ sa pamamagitan ng maingat na pagpili ng mga timbang na $x_j$ ng linear na kombinasyon. Upang kanselahin ang Trotter error para sa unang $l-1$ na termino (ang mga ito ay magbibigay ng pinakamalaking kontribusyon dahil katumbas ang mga ito ng mas mababang bilang ng Trotter step) sa expression para sa $\mu(t)$, ang mga coefficient na $x_j$ ay dapat matugunan ang mga sumusunod na equation:

$$ \sum_{j=1}^l x_j = 1 $$
$$ \sum_{j=1}^{l-1} \frac{x_j}{k_j^{n}} = 0 $$

na may $n=0, ... l-2$. Tinitiyak ng unang equation na walang bias sa constructed state na $\mu(t)$, habang tinitiyak ng pangalawang equation ang pagkansela ng mga Trotter error. Para sa higher-order PF, ang pangalawang equation ay nagiging $ \sum_{j=1}^{l-1} \frac{x_j}{k_j^{\eta}} = 0 $ kung saan ang $\eta = \chi + 2n$ para sa mga symmetric PF at $\eta = \chi + n$ kung hindi, na may $n=0, ..., l-2$. Ang resulta na error (Refs. [\[1\]](#references),[\[2\]](#references)) ay

$$ \epsilon = \mathcal{O} \left( \frac{t^{l+1}}{k_1^l} \right).$$

Ang pagtatukoy ng mga static MPF coefficient para sa isang ibinigay na hanay ng mga halaga ng $k_j$ ay katumbas ng paglutas ng linear system ng mga equation na tinukoy ng dalawang equation sa itaas para sa mga variable na $x_j$: $Ax=b$. Kung saan ang $x$ ang ating mga coefficient ng interes, ang $A$ ay isang matrix na nakasalalay sa $k_j$ at sa uri ng PF na ating ginagamit ($S$), at ang $b$ ay isang vector ng mga constraint. Partikular:

$$A_{0,j} = 1 $$
$$A_{i>0,j} = k_{j}^{-(\chi + s(i-1))}$$
$$b_0 = 1$$
$$b_{i>0} = 0 $$

kung saan ang $\chi$ ay ang ``order``, ang $s$ ay $2$ kung ang ``symmetric`` ay ``True`` at $1$ kung hindi, ang $k_{j}$ ay ang ``trotter_steps``, at ang $x$ ay ang mga variable na lulutas. Ang mga indeks na $i$ at $j$ ay nagsisimula sa $0$. Maaari rin nating visualisahin ito sa matrix form:

$$
A =
\begin{bmatrix}
A_{0,0} & A_{0,1} & A_{0,2} & ... \\
A_{1,0} & A_{1,1} & A_{1,2} & ...  \\
A_{2,0} & A_{2,1} & A_{2,2} & ...  \\
... & ... & ... & ...
\end{bmatrix} =
\begin{bmatrix}
1 & 1 & 1 & ... \\
k_{0}^{-(\chi + s(1-1))} & k_{1}^{-(\chi + s(1-1))} & k_{2}^{-(\chi + s(1-1))} & ... \\
k_{0}^{-(\chi + s(2-1))} & k_{1}^{-(\chi + s(2-1))} & k_{2}^{-(\chi + s(2-1))} & ... \\
... & ... & ... & ...
\end{bmatrix}
$$

at

$$
b =
\begin{bmatrix}
b_{0} \\
b_{1} \\
b_{2}  \\
...
\end{bmatrix} =
\begin{bmatrix}
1 \\
0 \\
0  \\
...
\end{bmatrix}
$$

Para sa karagdagang detalye, sumangguni sa dokumentasyon ng Linear System of Equations ([LSE](https://qiskit.github.io/qiskit-addon-mpf/stubs/qiskit_addon_mpf.static.LSE.html)).

Maaari tayong makahanap ng solusyon para sa $x$ nang analitiko bilang $x = A^{-1}b$; tingnan halimbawa ang Refs. [\[1\]](#references) o [\[2\]](#references).
Gayunpaman, ang eksaktong solusyong ito ay maaaring "ill-conditioned", na nagbubunga ng napakalaking L1-norm ng ating mga coefficient na $x$, na maaaring magdulot ng masamang pagganap ng MPF.
Sa halip, maaari ring makakuha ng tinantyang solusyon na nagpapaliit ng L1-norm ng $x$ upang subukang i-optimize ang gawi ng MPF.
##### I-set up ang LSE
Ngayon na napili na natin ang ating mga halaga ng $k_j$, kailangan muna nating itayo ang LSE, $Ax=b$ gaya ng ipinaliwanag sa itaas.
Ang matrix na $A$ ay nakasalalay hindi lamang sa $k_j$ kundi pati na rin sa ating pagpili ng PF, partikular ang _order_ nito.
Bukod pa rito, maaari mo ring isaalang-alang kung ang PF ay symmetric o hindi (tingnan ang [\[1\]](#references)) sa pamamagitan ng pagtatakda ng `symmetric=True/False`.
Gayunpaman, hindi ito kinakailangan, gaya ng ipinapakita ng Ref. [\[2\]](#references).

In [11]:
lse.A

array([[1.      , 1.      , 1.      ],
       [1.      , 0.25    , 0.0625  ],
       [1.      , 0.125   , 0.015625]])

In [12]:
lse.b

array([1., 0., 0.])

With the LSE in hand, we solve for the static coefficients $x_j$ via `lse.solve()` (this is the direct $x = A^{-1}b$ solution).

In [13]:
mpf_coeffs = lse.solve()
print(
    f"The static coefficients associated with the ansatze are: {mpf_coeffs}"
)

The static coefficients associated with the ansatze are: [ 0.04761905 -0.57142857  1.52380952]


Habang ang vector ng mga constraint na $b$ ay may mga sumusunod na elemento:
$$ b_{0} = 1 $$
$$ b_1 = b_2 = 0 $$

Kaya naman,

$$
b =
\begin{bmatrix}
1 \\
0 \\
0
\end{bmatrix}
$$

At katulad nito sa `lse`:

In [14]:
model_exact, coeffs_exact = setup_exact_problem(lse)
model_exact.solve()
print(coeffs_exact.value)

[ 0.04761905 -0.57142857  1.52380952]


In [15]:
print(
    "L1 norm of the exact coefficients:",
    np.linalg.norm(coeffs_exact.value, ord=1),
)

L1 norm of the exact coefficients: 2.1428571428556378


Ang `lse` object ay may mga pamamaraan para sa paghahanap ng mga static na coefficient na $x_j$ na natutugunan ang sistema ng mga equation.

In [16]:
model_approx, coeffs_approx = setup_sum_of_squares_problem(
    lse, max_l1_norm=1.5
)
model_approx.solve()
print(coeffs_approx.value)
print(
    "L1 norm of the approximate coefficients:",
    np.linalg.norm(coeffs_approx.value, ord=1),
)

[-1.10294118e-03 -2.48897059e-01  1.25000000e+00]
L1 norm of the approximate coefficients: 1.5


#### Dynamic MPF coefficients

The static MPF cancels Trotter-error terms in a Hamiltonian- and state-agnostic way, so it does not necessarily produce the smallest possible approximation error for a given Hamiltonian and initial state. The dynamic MPF (Refs. [\[2\]](#references), [\[3\]](#references)) instead finds time-dependent coefficients $x_i(t)$ that minimize the Frobenius-norm distance $\|\rho(t) - \mu^D(t)\|_F^2$ at each time $t$. As shown in the Background, this requires the overlap matrix $M_{ij}(t)$ between Trotter-evolved states and the overlap $L_i(t)$ with the exact state &mdash; both of which we estimate using tensor-network (TeNPy) backends in `qiskit_addon_mpf`.

To set up the dynamic LSE we need three ingredients:

1. An **approximate evolver factory** that the addon will run for each $k_j$ to produce $\rho_{k_j}(t)$ as an MPS/MPO. We build it from the layered structure of the order-$2$ Trotter circuit (one layer per `slice_by_depth`), wrapped as a `LayerwiseEvolver` with TeNPy truncation parameters.
2. An **exact evolver factory** that produces a high-accuracy reference $\rho(t)$. We use a small-time-step fourth-order Suzuki-Trotter circuit (`dt=0.1`, `order=4`) as a proxy for exact evolution.
3. An **identity factory** and an **initial-state MPS** that seed the TeNPy simulation.

The cell below constructs the approximate evolver factory.

In [17]:
# Create approximate time-evolution circuits
single_2nd_order_circ = generate_time_evolution_circuit(
    hamiltonian, time=1.0, synthesis=SuzukiTrotter(reps=1, order=order)
)
single_2nd_order_circ = pm.run(single_2nd_order_circ)  # collect XX and YY

# Find layers in the circuit
layers = slice_by_depth(single_2nd_order_circ, max_slice_depth=1)

# Create tensor network models
models = [
    LayerModel.from_quantum_circuit(layer, conserve="Sz") for layer in layers
]

# Create the time-evolution object
approx_factory = partial(
    LayerwiseEvolver,
    layers=models,
    options={
        "preserve_norm": False,
        "trunc_params": {
            "chi_max": 64,
            "svd_min": 1e-8,
            "trunc_cut": None,
        },
        "max_delta_t": 2,
    },
)

##### I-optimize ang $x$ gamit ang isang eksaktong modelo
Bilang alternatibo sa pagkalkula ng $x=A^{-1}b$, maaari mo ring gamitin ang [setup_exact_model](https://qiskit.github.io/qiskit-addon-mpf/stubs/qiskit_addon_mpf.static.setup_exact_model.html) upang itayo ang isang [cvxpy.Problem](https://www.cvxpy.org/api_reference/cvxpy.problems.html#cvxpy.Problem) instance na gumagamit ng LSE bilang mga constraint at ang optimal na solusyon nito ay magbubunga ng $x$.

Sa susunod na seksyon, magiging malinaw kung bakit umiiral ang interface na ito.

In [18]:
single_4th_order_circ = generate_time_evolution_circuit(
    hamiltonian, time=1.0, synthesis=SuzukiTrotter(reps=1, order=4)
)
single_4th_order_circ = pm.run(single_4th_order_circ)
exact_model_layers = [
    LayerModel.from_quantum_circuit(layer, conserve="Sz")
    for layer in slice_by_depth(single_4th_order_circ, max_slice_depth=1)
]

exact_factory = partial(
    LayerwiseEvolver,
    layers=exact_model_layers,
    dt=0.1,
    options={
        "preserve_norm": False,
        "trunc_params": {
            "chi_max": 64,
            "svd_min": 1e-8,
            "trunc_cut": None,
        },
        "max_delta_t": 2,
    },
)

Finally, we define an `identity_factory` that yields the initial MPO state and prepare the Néel initial state as an MPS that matches the lattice used by the layered Trotter model.

In [19]:
def identity_factory():
    return MPOState.initialize_from_lattice(models[0].lat, conserve=True)


mps_initial_state = MPS_neel_state(models[0].lat)

Bilang tagapagpahiwatig kung ang isang MPF na itinayo gamit ang mga coefficient na ito ay magbibigay ng magandang resulta, maaari nating gamitin ang L1-norm (tingnan din ang Ref. [\[1\]](#references)).

In [20]:
mpf_dynamic_coeffs_list = []
for t in trotter_times:
    print(f"Computing dynamic coefficients for time={t}")
    lse = setup_dynamic_lse(
        mpf_trotter_steps,
        t,
        identity_factory,
        exact_factory,
        approx_factory,
        mps_initial_state,
    )
    problem, coeffs = setup_frobenius_problem(lse)
    try:
        problem.solve()
        mpf_dynamic_coeffs_list.append(coeffs.value)
    except Exception as error:
        mpf_dynamic_coeffs_list.append(np.zeros(len(mpf_trotter_steps)))
        print(error, "Calculation Failed for time", t)
    print("")

Computing dynamic coefficients for time=0.5

Computing dynamic coefficients for time=0.6

Computing dynamic coefficients for time=0.7

Computing dynamic coefficients for time=0.7999999999999999

Computing dynamic coefficients for time=0.8999999999999999

Computing dynamic coefficients for time=0.9999999999999999

Computing dynamic coefficients for time=1.0999999999999999

Computing dynamic coefficients for time=1.1999999999999997

Computing dynamic coefficients for time=1.2999999999999998

Computing dynamic coefficients for time=1.4

Computing dynamic coefficients for time=1.4999999999999998



#### Combine Trotter expectation values with the MPF coefficients

Now we evaluate $\langle A \rangle_{\text{MPF}}(t) = \sum_j x_j \, \langle A \rangle_{k_j}(t)$ for each set of coefficients (static-exact, static-approximate, and dynamic), propagate the per-circuit standard errors, and plot the resulting time series against the exact-diagonalization curve.

In [21]:
sym = {1: "^", 2: "s", 4: "p"}
# Get expectation values at all times for each Trotter step
for k, step in enumerate(mpf_trotter_steps):
    trotter_curve, trotter_curve_error = [], []
    for trotter_expvals, trotter_stds in zip(
        mpf_expvals_all_times, mpf_stds_all_times
    ):
        trotter_curve.append(trotter_expvals[k])
        trotter_curve_error.append(trotter_stds[k])

    plt.errorbar(
        trotter_times,
        trotter_curve,
        yerr=trotter_curve_error,
        alpha=0.5,
        markersize=4,
        marker=sym[step],
        color="grey",
        label=f"{mpf_trotter_steps[k]} Trotter steps",
    )

# Get expectation values at all times for the static MPF with exact coeffs
exact_mpf_curve, exact_mpf_curve_error = [], []
for trotter_expvals, trotter_stds in zip(
    mpf_expvals_all_times, mpf_stds_all_times
):
    mpf_std = np.sqrt(
        sum(
            [
                (coeff**2) * (std**2)
                for coeff, std in zip(coeffs_exact.value, trotter_stds)
            ]
        )
    )
    exact_mpf_curve_error.append(mpf_std)
    exact_mpf_curve.append(trotter_expvals @ coeffs_exact.value)

plt.errorbar(
    trotter_times,
    exact_mpf_curve,
    yerr=exact_mpf_curve_error,
    markersize=4,
    marker="o",
    label="Static MPF - Exact",
    color="purple",
)


# Get expectation values at all times for the static MPF with approximate coeffs
approx_mpf_curve, approx_mpf_curve_error = [], []
for trotter_expvals, trotter_stds in zip(
    mpf_expvals_all_times, mpf_stds_all_times
):
    mpf_std = np.sqrt(
        sum(
            [
                (coeff**2) * (std**2)
                for coeff, std in zip(coeffs_approx.value, trotter_stds)
            ]
        )
    )
    approx_mpf_curve_error.append(mpf_std)
    approx_mpf_curve.append(trotter_expvals @ coeffs_approx.value)

plt.errorbar(
    trotter_times,
    approx_mpf_curve,
    yerr=approx_mpf_curve_error,
    markersize=4,
    marker="o",
    label="Static MPF - Approx",
    color="orange",
)


# Get expectation values at all times for the dynamic MPF
dynamic_mpf_curve, dynamic_mpf_curve_error = [], []
for trotter_expvals, trotter_stds, dynamic_coeffs in zip(
    mpf_expvals_all_times, mpf_stds_all_times, mpf_dynamic_coeffs_list
):
    mpf_std = np.sqrt(
        sum(
            [
                (coeff**2) * (std**2)
                for coeff, std in zip(dynamic_coeffs, trotter_stds)
            ]
        )
    )
    dynamic_mpf_curve_error.append(mpf_std)
    dynamic_mpf_curve.append(trotter_expvals @ dynamic_coeffs)

plt.errorbar(
    trotter_times,
    dynamic_mpf_curve,
    yerr=dynamic_mpf_curve_error,
    markersize=4,
    marker="o",
    label="Dynamic MPF",
    color="pink",
)


# Exact expectation values
plt.plot(
    exact_evolution_times,
    exact_expvals,
    color="red",
    linestyle="--",
    label="Exact time-evolution",
)

plt.title(f"$\\langle Z_{{{L//2-1}}} Z_{{{L//2}}} \\rangle$ vs time")
plt.xlabel("Time")
plt.ylabel("Expectation Value")
plt.legend(loc="upper center", bbox_to_anchor=(0.5, -0.2), ncol=2)
plt.grid(alpha=0.1)
plt.tight_layout()
plt.show()

<Image src="../docs/images/tutorials/multi-product-formula/extracted-outputs/35042576-0.avif" alt="Output of the previous code cell" />

##### I-optimize ang $x$ gamit ang isang tinantyang modelo
Maaaring mangyari na ang L1 norm para sa piniling hanay ng mga halaga ng $k_j$ ay masyadong mataas.
Kung ganoon ang kaso at hindi ka makapipili ng ibang hanay ng mga halaga ng $k_j$, maaari mong gamitin ang tinantyang solusyon sa LSE sa halip na eksaktong solusyon.

Upang gawin ito, gamitin lamang ang [setup_approximate_model](https://qiskit.github.io/qiskit-addon-mpf/stubs/qiskit_addon_mpf.static.setup_approximate_model.html) upang itayo ang ibang [cvxpy.Problem](https://www.cvxpy.org/api_reference/cvxpy.problems.html#cvxpy.Problem) instance, na naglilimita sa L1-norm sa isang piniling threshold habang pinipigilan ang pagkakaiba ng $Ax$ at $b$.

In [22]:
# -------------------------Step 1-------------------------
L = 50
coupling_map = CouplingMap.from_line(L, bidirectional=False)

# XXZ Hamiltonian with random couplings (Ref. [3])
np.random.seed(0)
even_edges = list(coupling_map.get_edges())[::2]
odd_edges = list(coupling_map.get_edges())[1::2]

Js = np.random.uniform(0.5, 1.5, size=L)
hamiltonian = SparsePauliOp(Pauli("I" * L))
for i, edge in enumerate(even_edges + odd_edges):
    hamiltonian += SparsePauliOp.from_sparse_list(
        [
            ("XX", (edge), 2 * Js[i]),
            ("YY", (edge), 2 * Js[i]),
            ("ZZ", (edge), 4 * Js[i]),
        ],
        num_qubits=L,
    )

observable = SparsePauliOp.from_sparse_list(
    [("ZZ", (L // 2 - 1, L // 2), 1.0)], num_qubits=L
)

total_time = 3
mpf_trotter_steps = [3, 4, 6]
order = 2
symmetric = True

# Static coefficients
lse = setup_static_lse(mpf_trotter_steps, order=order, symmetric=symmetric)
mpf_coeffs = lse.solve()
print(f"Static coefficients: {mpf_coeffs}")
print(f"L1 norm: {np.linalg.norm(mpf_coeffs, ord=1)}")

model_approx, coeffs_approx = setup_sum_of_squares_problem(
    lse, max_l1_norm=2.0
)
model_approx.solve()
print(f"Approximate coefficients: {coeffs_approx.value}")
print(f"L1 norm (approx): {np.linalg.norm(coeffs_approx.value, ord=1)}")

# -------------------------Dynamic coefficients-------------------------
single_2nd_order_circ = generate_time_evolution_circuit(
    hamiltonian, time=1.0, synthesis=SuzukiTrotter(reps=1, order=order)
)
single_2nd_order_circ = pm.run(single_2nd_order_circ)

layers = slice_by_depth(single_2nd_order_circ, max_slice_depth=1)
models = [
    LayerModel.from_quantum_circuit(layer, conserve="Sz") for layer in layers
]

approx_factory = partial(
    LayerwiseEvolver,
    layers=models,
    options={
        "preserve_norm": False,
        "trunc_params": {"chi_max": 64, "svd_min": 1e-8, "trunc_cut": None},
        "max_delta_t": 4,
    },
)

single_4th_order_circ = generate_time_evolution_circuit(
    hamiltonian, time=1.0, synthesis=SuzukiTrotter(reps=1, order=4)
)
single_4th_order_circ = pm.run(single_4th_order_circ)
exact_model_layers = [
    LayerModel.from_quantum_circuit(layer, conserve="Sz")
    for layer in slice_by_depth(single_4th_order_circ, max_slice_depth=1)
]

exact_factory = partial(
    LayerwiseEvolver,
    layers=exact_model_layers,
    dt=0.1,
    options={
        "preserve_norm": False,
        "trunc_params": {"chi_max": 64, "svd_min": 1e-8, "trunc_cut": None},
        "max_delta_t": 3,
    },
)


def identity_factory():
    return MPOState.initialize_from_lattice(models[0].lat, conserve=True)


mps_initial_state = MPS_neel_state(models[0].lat)

print(f"Computing dynamic coefficients for time={total_time}")
lse_dyn = setup_dynamic_lse(
    mpf_trotter_steps,
    total_time,
    identity_factory,
    exact_factory,
    approx_factory,
    mps_initial_state,
)
problem, coeffs_dyn = setup_frobenius_problem(lse_dyn)
try:
    problem.solve()
    mpf_dynamic_coeffs = coeffs_dyn.value
except Exception as error:
    mpf_dynamic_coeffs = np.zeros(len(mpf_trotter_steps))
    print(error, "Calculation Failed")

# -------------------------Step 1 (cont): Build circuits-------------------------
mpf_circuits = []
for k in mpf_trotter_steps:
    circuit = QuantumCircuit(L)
    circuit.x([i for i in range(L) if i % 2])
    trotter_circ = generate_time_evolution_circuit(
        hamiltonian,
        synthesis=SuzukiTrotter(reps=k, order=order),
        time=total_time,
    )
    circuit.compose(trotter_circ, qubits=range(L), inplace=True)
    mpf_circuits.append(circuit)

# Baseline "single deep circuit" comparison run with k=10 Trotter steps.
# Its two-qubit depth is deeper than the deepest MPF constituent (k_max=6) plus
# the overhead of running multiple circuits, pushing it into the noise-limited
# regime where MPF is expected to outperform. It does NOT target the MPF's effective
# Trotter error (which would require many more steps).
comp_circuit = QuantumCircuit(L)
comp_circuit.x([i for i in range(L) if i % 2])
trotter_circ = generate_time_evolution_circuit(
    hamiltonian,
    synthesis=SuzukiTrotter(reps=10, order=order),
    time=total_time,
)
comp_circuit.compose(trotter_circ, qubits=range(L), inplace=True)
mpf_circuits.append(comp_circuit)

Static coefficients: [ 0.42857143 -1.82857143  2.4       ]
L1 norm: 4.65714285714286
Approximate coefficients: [-0.4942491   0.40206845  1.09218065]
L1 norm (approx): 1.9884981979026675
Computing dynamic coefficients for time=3


We now optimize the circuits for the chosen backend. We use the Qiskit preset pass manager at `optimization_level=3`, which automatically selects a good set of physical qubits and routes each circuit onto the device topology.

In [23]:
# -------------------------Step 2-------------------------
service = QiskitRuntimeService()
# backend = service.least_busy(operational=True, simulator=False, min_num_qubits=L)
backend = service.backend("ibm_fez")
print(backend)

transpiler = generate_preset_pass_manager(
    optimization_level=3, backend=backend
)
transpiled_circuits = [transpiler.run(circ) for circ in mpf_circuits]

isa_observables = [
    observable.apply_layout(circ.layout) for circ in transpiled_circuits
]

<IBMBackend('ibm_fez')>


Tandaan na malaya kang lutasin ang optimization problem na ito ayon sa gusto mo, na nangangahulugang maaari mong baguhin ang optimization solver, ang mga convergence threshold nito, at iba pa.
Tingnan ang kaukulang gabay sa [How to use the approximate model](https://qiskit.github.io/qiskit-addon-mpf/how_tos/using_approximate_model.html).
#### Dynamic MPF coefficients

Sa nakaraang seksyon, nagpakilala tayo ng static MPF na nagpapabuti sa karaniwang Trotter approximation. Gayunpaman, hindi kinakailangan na mino-minimize ng static na bersyon na ito ang error sa approximation. Sa konkretong paraan, ang static MPF na tinatawag na $\mu^S(t)$, ay hindi ang pinakamainam na projection ng $\rho(t)$ sa subspace na sinasaklaw ng mga product-formula state na ${\rho_{k_i}(t)}_{i=1}^r$.

Upang matugunan ito, isinasaalang-alang natin ang isang dynamic MPF (ipinakikilala sa Ref. [\[2\]](#references) at ipinakita sa eksperimento sa Ref. [\[3\]](#references)) na talagang mino-minimize ang approximation error sa Frobenius norm. Pormal na sinasabi, nakatuon tayo sa pag-minimize ng

$$
\|\rho(t) - \mu^D(t)\|_F^2 \;=\; \mathrm{Tr}\bigl[ \left( \rho(t) - \mu^D(t)\right)^2 \bigr],
$$

kaugnay ng ilang mga coefficient na $x_i(t)$ sa bawat oras na $t$. Ang *pinakamainam* na projector sa Frobenius norm ay $\mu^D(t) \;=\; \sum_{i=1}^r x_i(t)\,\rho_{k_i}(t)$, at tinatawag nating $\mu^D(t)$ ang *dynamic* MPF. Sa pamamagitan ng paglalagay ng mga kahulugan sa itaas:

$$
\|\rho(t) - \mu^D(t)\|_F^2
\;=\; \\
= \mathrm{Tr}\bigl[ \left( \rho(t) - \mu^D(t)\right)^2 \bigr]
\;=\; \\
= \mathrm{Tr}\bigl[ \left( \rho(t) - \sum_{i=1}^r x_i(t)\,\rho_{k_i}(t) \right) \left(  \rho(t) - \sum_{j=1}^r x_j(t)\,\rho_{k_j}(t) \right) \bigr]
\;=\; \\
= 1 \;+\; \sum_{i,j=1}^r M_{i,j}(t)\,x_i(t)\,x_j(t)
\;-\;
2 \sum_{i=1}^r L_i^{\mathrm{exact}}(t)\,x_i(t),
$$

kung saan ang $M_{i,j}(t)$ ay ang *Gram matrix*, na tinukoy ng

$$
M_{i,j}(t) \;=\; \mathrm{Tr}\bigl[\rho_{k_i}(t)\,\rho_{k_j}(t)\bigr]
\;=\;
\bigl|\langle \psi_{\mathrm{in}} \!\mid S\bigl(t/k_i\bigr)^{-k_i}\,S\bigl(t/k_j\bigr)^{k_j} \!\mid \psi_{\mathrm{in}} \rangle \bigr|^2.
$$

at

$$
L_i^{\mathrm{exact}}(t) = \mathrm{Tr}[\rho(t)\,\rho_{k_i}(t)]
$$

ay kumakatawan sa overlap sa pagitan ng eksaktong estado na $\rho(t)$ at ng bawat product-formula approximation na $\rho_{k_i}(t)$. Sa mga praktikal na sitwasyon, ang mga overlap na ito ay maaaring masukat lamang nang may katumpakan dahil sa ingay o limitadong access sa $\rho(t)$.

Dito, ang $\lvert\psi_{\mathrm{in}}\rangle$ ay ang paunang estado, at ang $S(\cdot)$ ay ang operasyon na inilalapat sa product formula. Sa pamamagitan ng pagpili ng mga coefficient na $x_i(t)$ na nagmi-minimize sa expression na ito (at pangangasiwa ng approximate overlap data kapag hindi ganap na kilala ang $\rho(t)$), nakakakuha tayo ng "pinakamahusay" (sa kahulugan ng Frobenius norm) na dynamic approximation ng $\rho(t)$ sa loob ng MPF subspace. Ang mga dami na $L_i(t)$ at $M_{i,j}(t)$ ay maaaring kalkulahin nang mahusay gamit ang mga pamamaraan ng tensor network [\[3\]](#references). Nagbibigay ang MPF Qiskit addon ng ilang "backend" para isagawa ang kalkulasyon. Ipinapakita ng halimbawa sa ibaba ang pinaka-flexible na paraan para gawin ito, at ang [TeNPy layer-based backend](https://qiskit.github.io/qiskit-addon-mpf/apidocs/qiskit_addon_mpf.backends.tenpy_layers.html#module-qiskit_addon_mpf.backends.tenpy_layers) docs ay nagpapaliwanag din nang may detalye. Upang magamit ang pamamaraang ito, magsimula sa circuit na nagpapatupad ng nais na time-evolution at lumikha ng mga modelo na kumakatawan sa mga operasyong ito mula sa mga layer ng kaukulang circuit. Sa huli, lumilikha ng isang `Evolver` na object na maaaring gamitin para makabuo ng mga time-evolved na dami na $M_{i,j}(t)$ at $L_i(t)$. Nagsisimula tayo sa paglikha ng  `Evolver` na object na katumbas ng approximate time-evolution ([`ApproxEvolverFactory`](https://qiskit.github.io/qiskit-addon-mpf/apidocs/qiskit_addon_mpf.dynamic.html#qiskit_addon_mpf.dynamic.ApproxEvolverFactory)) na ipinapatupad ng mga circuit. Sa partikular, bigyang-pansin ang variable na `order` upang matiyak na magkatugma ang mga ito. Pansinin na sa pagbuo ng mga circuit na katumbas ng approximate time-evolution, gumagamit tayo ng placeholder na mga halaga para sa `time = 1.0` at sa bilang ng mga Trotter step (`reps=1`). Ang tamang mga approximating circuit ay ginagawa ng dynamic problem solver sa `setup_dynamic_lse`.

In [24]:
# -------------------------Step 3-------------------------
estimator = Estimator(mode=backend)
estimator.options.default_shots = 30000

# Error suppression/mitigation
estimator.options.dynamical_decoupling.enable = True
estimator.options.twirling.enable_gates = True
estimator.options.twirling.enable_measure = True
estimator.options.twirling.num_randomizations = "auto"
estimator.options.twirling.strategy = "active-accum"
estimator.options.resilience.measure_mitigation = True
estimator.options.experimental.execution_path = "gen3-turbo"

estimator.options.resilience.zne_mitigation = True
estimator.options.resilience.zne.noise_factors = (1, 1.2, 1.4)
estimator.options.resilience.zne.extrapolator = "linear"

estimator.options.environment.job_tags = ["TUT_MPF"]

job_50 = estimator.run(
    [
        (circ, observable)
        for circ, observable in zip(transpiled_circuits, isa_observables)
    ]
)

> **Warning:** Ang mga opsyon ng `LayerwiseEvolver` na nagtatakda ng mga detalye ng tensor network simulation ay dapat piliin nang maingat upang maiwasan ang pagtatayo ng isang ill-defined na optimization problem.
Pagkatapos ay itinatayo natin ang exact evolver (halimbawa, [`ExactEvolverFactory`](https://qiskit.github.io/qiskit-addon-mpf/apidocs/qiskit_addon_mpf.dynamic.html#qiskit_addon_mpf.dynamic.ExactEvolverFactory)), na nagbabalik ng isang [`Evolver`](https://qiskit.github.io/qiskit-addon-mpf/apidocs/qiskit_addon_mpf.backends.html#qiskit_addon_mpf.backends.Evolver) na object na nagkalkula ng tunay o "sanggunian" na time-evolution. Sa realidad, aapproximate natin ang eksaktong evolution sa pamamagitan ng paggamit ng higher-order Suzuki–Trotter formula o iba pang maaasahang pamamaraan na may maliit na time step. Sa ibaba, ina-approximate natin ang eksaktong time-evolved state gamit ang fourth-order Suzuki-Trotter formula na may maliit na time step na `dt=0.1`, na nangangahulugang ang bilang ng mga Trotter step na ginagamit sa oras na $t$ ay $k=t/dt$. Tinutukoy din natin ang ilang TeNPy-specific na truncation options para limitahan ang maximum bond dimension ng pinagbabatayang tensor network, pati na rin ang minimum singular values ng mga hiwalay na bond ng tensor network. Ang mga parameter na ito ay maaaring makaapekto sa katumpakan ng expectation value na kinakalkula gamit ang dynamic MPF coefficients, kaya't mahalaga na suriin ang hanay ng mga halaga upang mahanap ang pinakamainam na balanse sa pagitan ng oras ng pagkalkula at katumpakan. Tandaan na ang pagkalkula ng mga MPF coefficient ay hindi nakasalalay sa expectation value ng PF na nakuha mula sa pagpapatupad sa hardware, at samakatuwid ay maaari itong i-tune sa post-processing.

In [25]:
# -------------------------Step 4-------------------------
result = job_50.result()
evs = [res.data.evs for res in result]
std = [res.data.stds for res in result]

print(evs)
print(std)

[array(-0.07916195), array(-0.04479681), array(-0.2560756), array(-0.06045848)]
[array(0.04605538), array(0.10056336), array(0.14426151), array(0.04059092)]


In [26]:
exact_mpf_std = np.sqrt(
    sum([(coeff**2) * (std**2) for coeff, std in zip(mpf_coeffs, std[:3])])
)
print(
    "Exact static MPF expectation value: ",
    evs[:3] @ mpf_coeffs,
    "+-",
    exact_mpf_std,
)
approx_mpf_std = np.sqrt(
    sum(
        [
            (coeff**2) * (std**2)
            for coeff, std in zip(coeffs_approx.value, std[:3])
        ]
    )
)
print(
    "Approximate static MPF expectation value: ",
    evs[:3] @ coeffs_approx.value,
    "+-",
    approx_mpf_std,
)
dynamic_mpf_std = np.sqrt(
    sum(
        [
            (coeff**2) * (std**2)
            for coeff, std in zip(mpf_dynamic_coeffs, std[:3])
        ]
    )
)
print(
    "Dynamic MPF expectation value: ",
    evs[:3] @ mpf_dynamic_coeffs,
    "+-",
    dynamic_mpf_std,
)

Exact static MPF expectation value:  -0.5665938395816946 +- 0.3925273058119915
Approximate static MPF expectation value:  -0.25856647611537903 +- 0.164249927266166
Dynamic MPF expectation value:  -0.12667812062949296 +- 0.06059471006973169


In [27]:
sym = {3: "^", 4: "s", 6: "p"}
for k, step in enumerate(mpf_trotter_steps):
    plt.errorbar(
        k,
        evs[k],
        yerr=std[k],
        alpha=0.5,
        markersize=4,
        marker=sym[step],
        color="grey",
        label=f"{mpf_trotter_steps[k]} Trotter steps",
    )

plt.errorbar(
    3,
    evs[-1],
    yerr=std[-1],
    alpha=0.5,
    markersize=8,
    marker="x",
    color="blue",
    label="10 Trotter steps",
)

plt.errorbar(
    4,
    evs[:3] @ mpf_coeffs,
    yerr=exact_mpf_std,
    markersize=4,
    marker="o",
    color="purple",
    label="Static MPF",
)

plt.errorbar(
    5,
    evs[:3] @ coeffs_approx.value,
    yerr=approx_mpf_std,
    markersize=4,
    marker="o",
    color="orange",
    label="Approximate static MPF",
)

plt.errorbar(
    6,
    evs[:3] @ mpf_dynamic_coeffs,
    yerr=dynamic_mpf_std,
    markersize=4,
    marker="o",
    color="pink",
    label="Dynamic MPF",
)

exact_obs = -0.24384471447172074  # Calculated via Tensor Network calculation
plt.axhline(
    y=exact_obs, linestyle="--", color="red", label="Exact time-evolution"
)

plt.title(
    f"$\\langle Z_{{{L//2-1}}} Z_{{{L//2}}} \\rangle$ at time {total_time} for the different methods"
)
plt.xlabel("Method")
plt.ylabel("Expectation Value")
plt.legend(loc="upper center", bbox_to_anchor=(0.5, -0.2), ncol=2)
plt.grid(alpha=0.1)
plt.tight_layout()
plt.show()

<Image src="../docs/images/tutorials/multi-product-formula/extracted-outputs/64360d85-0.avif" alt="Output of the previous code cell" />

A few observations about the hardware results above:

- **Going deeper is not free on hardware.** The single-circuit baselines tell the story directly: the $k = 6$ circuit is essentially exact ($-0.256$ versus the reference $-0.244$), yet the deeper $k = 10$ baseline is *worse* ($-0.061$, off by $\sim 0.18$), not better. Once the Trotter error is already small, adding steps mostly deepens the circuit and accumulates more gate noise and decoherence. This is precisely the regime MPFs are built for: reach the accuracy of a deep circuit using only shallow constituents.

- **A small-norm MPF beats the deep single circuit.** The approximate-static MPF (capped at $\|x\|_1 \approx 2$) lands at $-0.259$, within $\sim 0.015$ of the reference and far closer than the $k = 10$ baseline. The dynamic MPF ($-0.127$) also comfortably beats that baseline. Both combine only the shallow $k_j = [3, 4, 6]$ circuits, yet recover an answer the deep single circuit could not.

- **Coefficient norm matters more than mathematical optimality.** The exact-static MPF has $\|x\|_1 = 4.66$ and is the *worst* estimator of all ($-0.567$, off by more than $0.3$): the large coefficient norm amplifies the residual gate noise, decoherence, and ZNE error on each $\langle A \rangle_{k_j}$ by roughly the same factor, overwhelming the Trotter-error cancellation it buys. Capping the norm (the approximate-static solver, $\|x\|_1 \approx 2$) removes this overwhelm and gives the best estimate &mdash; even though its coefficients no longer cancel the leading Trotter error exactly.

- **Individual shallow circuits can still be competitive.** The lone $k = 6$ constituent ($-0.256$) is itself essentially exact here &mdash; on this run it is even marginally closer than the approximate-static MPF. The catch is that you do not know in advance *which* single $k$ sits in the sweet spot of "converged but not yet noise-limited," and the safe-looking choice of simply going deeper ($k = 10$) to guarantee Trotter convergence is exactly the one that fails. The MPF gives a principled combination of shallow circuits that does not require guessing the right depth.

The practical takeaway is that on hardware, MPFs should be paired with strong error mitigation on each individual $\langle A \rangle_{k_j}$, the coefficient $L_1$-norm should be kept modest (use the approximate solver, or the dynamic MPF), and the Trotter steps $k_j$ should be chosen so that $t/k_{\min} \lesssim 1$ &mdash; here $k_{\min} = 3$ at $t = 3$ gives $t/k_{\min} = 1$, keeping the constituents inside the convergent regime where the leading-error model the static MPF relies on is valid. With those choices, the small-norm MPFs here match a converged single circuit while the naive "just go deeper" baseline does not, recovering the depth-versus-accuracy advantage shown in Ref. [\[3\]](#references). Note also that individual runs are noisy &mdash; on a different submission of the same job (or a different backend), the exact ordering can shift; the robust trends are that small-$\|x\|_1$ MPFs do well, the large-$\|x\|_1$ exact-static MPF is amplified by hardware noise, and the over-deep single circuit is noise-limited.

## Next steps
<Admonition type="tip" title="Recommendations">

If you found this work interesting, you might be interested in the following material:
- [How to choose the Trotter steps for an MPF](https://qiskit.github.io/qiskit-addon-mpf/how_tos/choose_trotter_steps.html) — practical guidance on selecting $k_j$ values to avoid instabilities
- [How to use the approximate model](https://qiskit.github.io/qiskit-addon-mpf/how_tos/using_approximate_model.html) — tuning the $L_1$-norm constraint and solver options for the approximate static MPF
- [`qiskit-addon-mpf` API reference](https://qiskit.github.io/qiskit-addon-mpf/) — full documentation for static, dynamic, and backend modules
</Admonition>

## References

\[1] Vazquez, A. C., Egger, D. J., Ochsner, D., & Woerner, S. Well-conditioned multi-product formulas for hardware-friendly Hamiltonian simulation. [Quantum, 7, 1067 (2023)](https://quantum-journal.org/papers/q-2023-07-25-1067/)

\[2] Zhuk, S., Robertson, N. F., & Bravyi, S. Trotter error bounds and dynamic multi-product formulas for Hamiltonian simulation. [Physical Review Research, 6(3), 033309 (2024)](https://journals.aps.org/prresearch/abstract/10.1103/PhysRevResearch.6.033309)

\[3] Robertson, N. F., et al. Tensor network enhanced dynamic multiproduct formulas. [arXiv:2407.17405 (2024)](https://arxiv.org/abs/2407.17405)